In [0]:
from pyspark.sql.functions import expr , col

members_df = spark.table("dev.spark_db.members").where(" member_id > 0 ").select("member_id" , "first_name" , "last_name").alias("m")

bookings_df = spark.table("dev.spark_db.bookings").alias("b")

In [0]:
members_df.lateralJoin(bookings_df.where("m.member_id == b.member_id").orderBy(col("start_time").desc()).limit(1)
                       , None , 'left')\
            .select("m.member_id" , "m.first_name" , "m.last_name" , "b.facility_id" , "b.start_time" , "b.slots").alias("mb")\
            .join(spark.table("dev.spark_db.facilities").alias("f") , on = expr("mb.facility_id == f.facility_id") , how = 'left')\
            .select("mb.member_id" , "mb.first_name" , "mb.last_name" , "f.facility_name" , "mb.start_time" , "mb.slots")\
            .display()

In [0]:
from pyspark.sql.functions import expr

students_df = spark.table("dev.spark_db.offline_var_students").alias("s")

results_df = (
    students_df.lateralJoin(
        spark.tvf.variant_explode("s.Skills")
            .selectExpr(
                "cast(value:Skill as string) as skill", 
                "cast(value:YearsOfExperience as int) as yearofexperience"
            )
            .where("skill like '%Spark%' and yearofexperience > 1")
    )
)
# spark.tvf.variant_explode("s.skills") - > 3 columns  position of the element  key of the join and value 

results_df.display()